# MixIT-MSS - Separation & Listening

Notebook to run inference with a trained MixIT / BS-Locoformer model and listen to
(or save) the separated tracks.

It covers **both scenarios**:
- **RAW** - the pre-training checkpoint (`pretrained_mixit.pth`): emits the *N* (e.g. 12)
  unlabeled MixIT channels. Diagnostic: hear *what the unsupervised model learned*.
- **FINE-TUNED** - the fine-tuning checkpoint (`finetuned_musdb.pth`): emits the 4 VDBO
  stems (vocals / drums / bass / other), using the saved channel map.

Target data: the **MUSDB18 test split**.

## 1. Setup & config

In [ ]:
# If you did NOT `pip install -e .`, uncomment to add the project root to the path:
# import sys; sys.path.insert(0, "/nas/home/macerbi/mixit-mss")

import os
import torch
import torchaudio
from IPython.display import Audio, display

from mixit_mss.inference import (
    load_separator, separate, separate_waveform,
    separate_musdb_test, save_stems,
)
from mixit_mss.channel_selection import STEMS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Checkpoints (edit to your files)
PRETRAIN_CKPT  = "pretrained_mixit.pth"     # RAW scenario
FINETUNE_CKPT  = "finetuned_musdb.pth"      # FINE-TUNED scenario (may not exist yet)

# MUSDB18 test split root (standard layout: <root>/<track>/mixture.wav)
MUSDB_TEST_ROOT = "/nas/home/macerbi/dataset/musdb18hq/test"

# Where to write separated wavs
OUT_ROOT = "sep_out"

# IMPORTANT: these MUST match the values used at TRAINING time. If you trained with
# reduced settings for memory (e.g. --n_layers 4), change them here too, otherwise
# the weights won't load correctly.
MODEL_KW = dict(
    n_srcs=12,          # --n_srcs
    n_channels=2,       # stereo
    n_layers=6,         # --n_layers
    emb_dim=128,        # --emb_dim
    sr=44100,           # --sr
    stft_size=2048,     # --stft_size
    hop_length=512,     # --hop_length
)

# Inference (overlap-add) settings
CHUNK_SECONDS = 6.0     # window length; lower if you hit OOM at inference
OVERLAP = 0.5           # 50% overlap between chunks
print("config set")

## 2. Load a separator

`load_separator` auto-detects the scenario from the checkpoint contents:
- a plain state_dict $\rightarrow$ RAW (pre-trained)
- a dict with a `channel_map` $\rightarrow$ FINE-TUNED.

In [ ]:
# Pick ONE checkpoint to load. Start with whichever you have trained.
CKPT = PRETRAIN_CKPT          # or: CKPT = FINETUNE_CKPT

sep = load_separator(CKPT, device=device, **MODEL_KW)
print("loaded:", CKPT)
print("fine-tuned (VDBO stems)?", sep.is_finetuned)
if sep.is_finetuned:
    print("channel map (stem -> channel):", sep.channel_map)
else:
    print(f"RAW mode: model emits {sep.n_srcs} unlabeled channels")

## 3. Separate a single MUSDB test track and listen

We take one track's `mixture.wav`, separate it, and play the results inline.

- In **FINE-TUNED** mode you get 4 named stems.
- In **RAW** mode you get N channels named `src00..srcNN` — expect some
  over-separation (an instrument split across channels); this is diagnostic.

In [ ]:
# List available test tracks
tracks = sorted(d for d in os.listdir(MUSDB_TEST_ROOT)
                if os.path.isdir(os.path.join(MUSDB_TEST_ROOT, d)))
print(f"{len(tracks)} test tracks. First few:")
for t in tracks[:5]:
    print("  ", t)

In [ ]:
# Choose a track and separate it
track = tracks[0]
mix_path = os.path.join(MUSDB_TEST_ROOT, track, "mixture.wav")
print("separating:", track)

stems = separate(sep, mix_path, chunk_seconds=CHUNK_SECONDS, overlap=OVERLAP)
print("got", len(stems), "outputs:", list(stems.keys()))

In [ ]:
# Listen to the original mixture first (downmixed to mono just for the player)
mix, sr = torchaudio.load(mix_path)
print("MIXTURE")
display(Audio(mix.mean(0).numpy(), rate=sr))

In [ ]:
# Listen to each separated output.
# (stems[name] is a [C, L] tensor; we downmix to mono for the inline player.)
for name, wav in stems.items():
    print(name.upper())
    display(Audio(wav.mean(0).numpy(), rate=sep.sr))

### RAW mode helper - find which channels are "loud"

In RAW mode with 12 channels, many may be near-silent. This ranks channels by
energy so you can focus on the active ones.

In [ ]:
if not sep.is_finetuned:
    energies = {name: float((wav ** 2).mean()) for name, wav in stems.items()}
    ranked = sorted(energies.items(), key=lambda kv: kv[1], reverse=True)
    print("channels by energy (loudest first):")
    for name, e in ranked:
        print(f"  {name}: {e:.6f}")
    # play the top 4 loudest channels
    print("\nTop-4 loudest channels:")
    for name, _ in ranked[:4]:
        print(name.upper())
        display(Audio(stems[name].mean(0).numpy(), rate=sep.sr))
else:
    print("Fine-tuned model: stems are already the 4 VDBO sources.")

## 4. Save the separated tracks to disk

In [ ]:
# Write this track's stems to OUT_ROOT/<track>/<name>.wav
out_dir = os.path.join(OUT_ROOT, track)
paths = save_stems(stems, out_dir, sr=sep.sr)
print("written:")
for name, p in paths.items():
    print("  ", p)

## 5. Batch: separate the whole MUSDB test split

`separate_musdb_test` runs over every track and writes `OUT_ROOT/<track>/<stem>.wav`.
Use `limit` to do only the first few while experimenting; remove it for the full split.

In [ ]:
results = separate_musdb_test(
    sep, MUSDB_TEST_ROOT, OUT_ROOT,
    limit=3,                       # set to None for all test tracks
    chunk_seconds=CHUNK_SECONDS, overlap=OVERLAP,
)
print("\ndone:", len(results), "tracks")
for track, out_dir in results:
    print("  ", track, "->", out_dir)

## 6. (Optional) Separate an arbitrary audio file

Not limited to MUSDB — point it at any stereo file.

In [ ]:
# my_file = "/path/to/any/song.wav"
# stems = separate(sep, my_file, chunk_seconds=CHUNK_SECONDS, overlap=OVERLAP)
# for name, wav in stems.items():
#     print(name.upper()); display(Audio(wav.mean(0).numpy(), rate=sep.sr))
# save_stems(stems, "sep_out/my_file", sr=sep.sr)